# Tutorial 2 — Analysis

Companion notebook for Part 2 of the tutorial. It assumes all five
`tutorial_02` scenarios have been solved and analysed
(`tools/smk --configfile config/tutorial/02_demand_response.yaml -- analyze_all_scenarios`).

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path("..", "..").resolve()
scenarios = ["reference", "ghg_mid", "ghg_high", "ghg_high_stiff", "ghg_high_fixed"]


def load_analysis(filename: str, which=None) -> pd.DataFrame:
    """Concatenate a per-scenario analysis parquet for tutorial_02."""
    results = project_root / "results" / "tutorial_02" / "analysis"
    return pd.concat(
        pd.read_parquet(results / f"scen-{s}" / filename).assign(scenario=s)
        for s in (which or scenarios)
    )

## What the calibration inferred about consumers

The fitted demand wedges live in the tutorial's artefact set. For `demand`
rows the intercept is *minus* the willingness to pay, in bn USD/Mt --
numerically the same as USD/kg. The ranking should look economically
sensible: high-value animal products at the top, staples near the bottom.

In [ ]:
curves = pd.read_csv(
    project_root / "data/curated/calibration/tutorial-02/market_response.csv"
)
demand = curves[curves["component"] == "demand"].assign(
    wtp_usd_per_kg=lambda df: -df["intercept"]
)
demand.sort_values("wtp_usd_per_kg", ascending=False).head(10)[
    ["mr_group", "wtp_usd_per_kg", "slope_basis"]
].reset_index(drop=True)

## Does the diet actually move?

Global food-group consumption across the carbon-price ladder. The `reference`
column doubles as a calibration check: it should match observed 2020
consumption almost exactly.

In [ ]:
consumption = load_analysis("food_group_consumption.parquet")
by_group = (
    consumption.groupby(["scenario", "food_group"])["consumption_mt"]
    .sum()
    .unstack("food_group")
    .reindex(scenarios)
)

ax = by_group.plot.bar(stacked=True, ylabel="Global consumption (Mt)", rot=20)
ax.set_title("Food-group consumption")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()

## The two sides of the market

`supply_response` and `demand_response` are the deviation costs of moving
production and consumption off their observed patterns; `ghg_cost` is the
force pushing them. Note how both response terms grow with the carbon price,
and how `demand_response` vanishes in the fixed-diet scenario.

In [ ]:
breakdown = (
    load_analysis("objective_breakdown.parquet")
    .set_index("scenario")
    .reindex(scenarios)
)
columns = [
    "ghg_cost",
    "supply_response",
    "demand_response",
    "crop_production",
    "animal_production",
]
breakdown[[c for c in columns if c in breakdown.columns]].round(2)

## The value of dietary flexibility

Net emissions at the same $200/tCO2-eq price under three demand regimes:
fully elastic (`ghg_high`), half-elasticity (`ghg_high_stiff`), and frozen at
the observed diet (`ghg_high_fixed`). The elastic-vs-fixed gap is the
demand-side mitigation potential; the stiff case shows how much of it hinges
on the elasticity assumption.

In [ ]:
emissions = load_analysis("net_emissions.parquet")
totals = (
    emissions.groupby("scenario")["mtco2eq"]
    .sum()
    .reindex(["reference", "ghg_high", "ghg_high_stiff", "ghg_high_fixed"])
)
print(totals.round(1))

ax = totals.plot.bar(ylabel="Net GHG (MtCO2eq)", rot=20)
ax.set_title("Same carbon price, three demand regimes")
plt.tight_layout()